# Training a Toxicity Classifier on Civil Comments

This notebook trains a BERT-based toxicity classifier on the Civil Comments dataset from Kaggle.
The trained model serves as the "proxy" metric for the estimate-level adjustment methodology.

## Prerequisites
- Download the Civil Comments dataset from [Kaggle](https://www.kaggle.com/datasets/arabel1a/wilds-civilcomments)
- Install dependencies: `pip install -e ".[ml]"`

In [ ]:
%local-changes

In [ ]:
import gc

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
from transformers import AutoModel, AutoTokenizer

In [ ]:
from estimate_level_adjustment.kaggle_dataset import load_kaggle_dataset

civil_comments = load_kaggle_dataset(
    dataset_name="CivilComments",
    data_dir="KaggleDatasets/CivilComments",
    kaggle_url="https://www.kaggle.com/datasets/arabel1a/wilds-civilcomments",
)
zip_path = "KaggleDatasets/CivilComments/CivilComments.zip"
civil_comments.extract_downloaded_zip(zip_path)
civil_comments.list_csv_files()
civil_comments.load_all_csvs()
df = civil_comments.load_csv('civilcomments_v1.0/all_data_with_identities.csv')
df.head()

In [ ]:
# define target
df['rating'] = df['rating'].apply(lambda x: x=='rejected')

primary_binary = 'rating'
df['target'] = df[primary_binary]

# Loading the datasets
train_df = df[df['split']=='train']
test_df = df[df['split']=='test']
val_df = df[df['split']=='val']

In [ ]:
from estimate_level_adjustment.civil_comments_model import (
    load_toxicity_model,
    predict_toxicity_text,
    train_toxicity_model,
)

# Train the model
model, tokenizer, history = train_toxicity_model(
    train_df,
    model_path="bert-base-uncased",
    text_column="comment_text",
    label_column="target",
    batch_size=32,
    epochs=5,
    learning_rate=3e-5,
    max_length=256,
    patience_epochs=2,
    save_path="toxicity_model.pt",
)

In [ ]:
from estimate_level_adjustment.civil_comments_model import (
    predict_toxicity_text,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
test_texts = test_df['comment_text'].values
test_preds = []
for test_text in test_texts:
    test_pred = predict_toxicity_text(model, tokenizer, test_text, device)
    test_preds.append(test_pred)

In [ ]:
test_df_results = pd.merge(test_df, pd.DataFrame(test_preds))
test_df_results["row_id"] = test_df_results["id"]

# Create metrics dataframe in long format
test_df_proxy = test_df_results[["row_id", "is_toxic"]].copy()
test_df_proxy["metric"] = test_df_proxy["is_toxic"].apply(lambda x: int(x))
test_df_proxy["model_name"] = "proxy"
test_df_proxy["weight"] = 1

test_df_primary = test_df_results[["row_id", "target"]].copy()
test_df_primary["metric"] = test_df_primary["target"].apply(lambda x: int(x))
test_df_primary["model_name"] = "primary"
test_df_primary["weight"] = 1

metrics_df = pd.concat(
    [
        test_df_proxy[["row_id", "model_name", "metric", "weight"]],
        test_df_primary[["row_id", "model_name", "metric", "weight"]],
    ],
    axis=0,
)

entities_df = test_df_results[
    ["row_id", "created_date", "publication_id", "parent_id", "article_id"]
]

# Save to CSV for use in the results notebook
metrics_df.to_csv("civil_comments_metrics.csv", index=False)
entities_df.to_csv("civil_comments_entities.csv", index=False)
print(f"Saved {len(metrics_df)} metric rows and {len(entities_df)} entity rows")

## Next Steps

The saved CSV files (`civil_comments_metrics.csv` and `civil_comments_entities.csv`) 
are used by the `civic_comments_results.ipynb` notebook to apply the estimate-level 
adjustment methodology.